# Summary_Day8.ipynb  
## 이진분류 · BCE · 평가 지표 · 실무형 분류 파이프라인

이번 8차시는 **이진 분류 Binary Classification**를 중심으로 정리합니다.

이번 파일부터는 각 주요 함수마다 다음 내용을 함께 정리합니다.

```text
1. 함수가 하는 일
2. 기본 사용 형태
3. 괄호 안 인자의 의미
4. 실습 코드에서 왜 필요한지
5. 언제 쓰면 되는지
```

핵심 목표:

1. 회귀와 이진 분류의 차이 이해
2. Sigmoid 함수와 0.5 기준 분류 이해
3. BCELoss와 BCEWithLogitsLoss 차이 이해
4. Iris 데이터로 이진 로지스틱 회귀 구현
5. 학습/검증 데이터 분할
6. Loss와 Accuracy 학습 곡선 확인
7. 결정 경계 Decision Boundary 시각화
8. 사기 거래 탐지처럼 클래스 불균형이 있는 문제 다루기
9. Confusion Matrix, Precision, Recall, F1, ROC-AUC 이해
10. pos_weight로 소수 클래스 보정하기
11. 당뇨병 예측 실무형 구조: Config, DataPreprocessor, Trainer, EarlyStopping
12. 고객 이탈 예측 실무형 구조: Pipeline, ColumnTransformer, OneHotEncoder, threshold tuning, feature importance

핵심 흐름:

```text
데이터 준비 → 분할 → 전처리 → 모델 정의 → 손실 함수 → 학습 → 평가 지표 확인 → 임계값 조정
```

## 1. 라이브러리 준비

이번 실습에서는 PyTorch와 scikit-learn을 함께 사용합니다.

### 함수 사용법

| 함수/모듈 | 기본 사용 형태 | 언제 쓰는가 |
|---|---|---|
| `train_test_split(X, y, test_size=..., stratify=...)` | 데이터를 학습/테스트로 나눔 | 모델 평가용 데이터 분리 |
| `StandardScaler().fit_transform(X_train)` | 평균 0, 표준편차 1로 표준화 | 수치형 feature 스케일 맞춤 |
| `nn.BCELoss()` | 확률값과 정답 비교 | 모델 마지막에 Sigmoid가 있을 때 |
| `nn.BCEWithLogitsLoss()` | raw logit과 정답 비교 | 모델 마지막에 Sigmoid가 없을 때 권장 |
| `confusion_matrix(y_true, y_pred)` | 혼동행렬 계산 | 분류 결과 상세 확인 |
| `roc_auc_score(y_true, y_prob)` | ROC-AUC 계산 | 임계값 전체 기준 성능 평가 |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_iris, make_classification, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    RocCurveDisplay
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

%matplotlib inline

np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("device:", device)
print("PyTorch:", torch.__version__)

## 2. 이진 분류 개념

이진 분류는 데이터를 두 그룹 중 하나로 나누는 문제입니다.

예:

```text
정상 거래 / 사기 거래
당뇨 위험 낮음 / 높음
고객 유지 / 고객 이탈
스팸 아님 / 스팸
```

회귀와의 차이:

| 문제 | 출력 |
|---|---|
| 회귀 | 연속적인 숫자 |
| 이진 분류 | 0 또는 1 |

이진 분류에서는 모델 출력값을 확률처럼 해석하기 위해 Sigmoid를 사용합니다.

In [ ]:
examples = {
    "Fraud Detection": "정상 거래(0) / 사기 거래(1)",
    "Diabetes Risk": "저위험(0) / 고위험(1)",
    "Customer Churn": "유지(0) / 이탈(1)",
    "Spam Filter": "정상 메일(0) / 스팸 메일(1)"
}

for task, desc in examples.items():
    print(f"{task}: {desc}")

## 3. Sigmoid 함수

Sigmoid는 어떤 숫자든 0과 1 사이 값으로 바꿉니다.

공식:

```text
sigmoid(x) = 1 / (1 + exp(-x))
```

### 함수 사용법

```python
torch.sigmoid(x)
```

- `x`: Tensor 입력값
- 반환값: 0과 1 사이 Tensor

이진 분류에서는 보통 다음처럼 해석합니다.

```text
sigmoid(output) >= 0.5 → class 1
sigmoid(output) < 0.5  → class 0
```

In [ ]:
x_sig = torch.linspace(-6, 6, 100)
y_sig = torch.sigmoid(x_sig)

plt.plot(x_sig, y_sig)
plt.axhline(0.5, linestyle="--")
plt.axvline(0.0, linestyle="--")
plt.xlabel("x")
plt.ylabel("sigmoid(x)")
plt.title("Sigmoid Function")
plt.show()

그래프 해석:

- 입력이 0이면 출력은 0.5입니다.
- 입력이 클수록 1에 가까워집니다.
- 입력이 작을수록 0에 가까워집니다.
- 따라서 0.5를 기준으로 class 0/1을 나눌 수 있습니다.

## 4. Iris 데이터 준비

Iris 데이터는 붓꽃의 꽃받침/꽃잎 길이와 너비를 담은 데이터입니다.

이번 실습에서는 이진 분류에 집중하기 위해:

```text
전체 3개 품종 중 앞의 2개 품종만 사용
전체 4개 feature 중 앞의 2개 feature만 사용
```

### 함수 사용법: `load_iris()`

```python
iris = load_iris()
X = iris.data
y = iris.target
```

- `iris.data`: 입력 feature
- `iris.target`: 정답 label

In [ ]:
iris = load_iris()

x_org = iris.data
y_org = iris.target

x_data = x_org[:100, :2]
y_data = y_org[:100]

print("원본 데이터:", x_org.shape, y_org.shape)
print("이진 분류용 데이터:", x_data.shape, y_data.shape)
print("클래스 종류:", np.unique(y_data))

코드 설명:

- `x_org[:100, :2]`: 앞 100개 데이터, 앞 2개 feature만 선택합니다.
- `y_org[:100]`: 앞 100개 정답만 선택합니다.
- 결과적으로 class 0과 class 1만 남습니다.

## 5. Train/Test 데이터 분할

모델이 학습한 데이터에만 잘 맞는지, 새로운 데이터에도 잘 맞는지 확인하려면 데이터를 나눠야 합니다.

### 함수 사용법: `train_test_split()`

```python
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)
```

- `X`: 입력 데이터
- `y`: 정답 데이터
- `test_size`: 테스트 데이터 비율
- `random_state`: 결과 재현을 위한 랜덤 고정값
- `stratify=y`: train/test의 클래스 비율을 원본과 비슷하게 유지

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x_data,
    y_data,
    train_size=70,
    test_size=30,
    random_state=42,
    stratify=y_data
)

print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

`stratify`를 쓰면 class 0과 class 1의 비율이 train/test에서 비슷하게 유지됩니다.

이진 분류에서는 클래스 비율이 중요하므로 자주 사용합니다.

## 6. Iris 산점도 확인

두 class가 feature 공간에서 어떻게 나뉘는지 산점도로 봅니다.

In [ ]:
x_t0 = x_train[y_train == 0]
x_t1 = x_train[y_train == 1]

plt.scatter(x_t0[:, 0], x_t0[:, 1], marker="x", label="class 0")
plt.scatter(x_t1[:, 0], x_t1[:, 1], marker="o", label="class 1")
plt.xlabel("sepal_length")
plt.ylabel("sepal_width")
plt.title("Iris Binary Classification Data")
plt.legend()
plt.show()

그래프 해석:

- 점의 위치가 feature입니다.
- marker `x`와 `o`는 서로 다른 class입니다.
- 로지스틱 회귀는 이 두 그룹을 나누는 직선, 즉 결정 경계를 학습합니다.

## 7. Tensor 변환

PyTorch 모델에 넣으려면 NumPy 배열을 Tensor로 바꿔야 합니다.

### 함수 사용법

```python
torch.tensor(array).float()
```

- `torch.tensor(array)`: 배열을 Tensor로 변환
- `.float()`: 실수형 Tensor로 변환

정답 label은 BCE 손실에 맞게 `[N, 1]` 형태로 바꿉니다.

```python
labels.view(-1, 1)
```

In [ ]:
inputs = torch.tensor(x_train).float()
labels = torch.tensor(y_train).float().view(-1, 1)

inputs_test = torch.tensor(x_test).float()
labels_test = torch.tensor(y_test).float().view(-1, 1)

print("inputs:", inputs.shape)
print("labels:", labels.shape)
print("inputs_test:", inputs_test.shape)
print("labels_test:", labels_test.shape)

BCE 계열 손실함수에서는 예측값과 정답값 shape이 같아야 합니다.

그래서 정답도 `[N, 1]` 형태로 맞춥니다.

## 8. BCELoss 방식 모델 정의

기존 방식은 모델 안에서 Sigmoid까지 적용하고, 손실함수는 `BCELoss`를 사용합니다.

구조:

```text
Linear → Sigmoid → BCELoss
```

### 함수 사용법: `nn.Linear()`

```python
nn.Linear(in_features, out_features)
```

- `in_features`: 입력 feature 개수
- `out_features`: 출력 feature 개수

### 함수 사용법: `nn.Sigmoid()`

```python
self.sigmoid = nn.Sigmoid()
x = self.sigmoid(x)
```

- 입력값을 0~1 확률값으로 변환합니다.

In [ ]:
class IrisBCENet(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)
        self.sigmoid = nn.Sigmoid()

        self.l1.weight.data.fill_(1.0)
        self.l1.bias.data.fill_(1.0)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.sigmoid(x1)
        return x2

n_input = inputs.shape[1]
n_output = 1

bce_net = IrisBCENet(n_input, n_output)

print(bce_net)

코드 설명:

- `n_input = 2`: Iris feature를 2개 사용합니다.
- `n_output = 1`: class 1일 확률 하나를 출력합니다.
- `forward()`는 `Linear → Sigmoid` 순서로 진행됩니다.

## 9. BCELoss 학습 준비

### 함수 사용법: `nn.BCELoss()`

```python
criterion = nn.BCELoss()
loss = criterion(pred_prob, target)
```

- `pred_prob`: Sigmoid를 지난 0~1 확률값
- `target`: 0 또는 1 정답값
- 주의: raw logit을 넣으면 안 됩니다.

### 함수 사용법: `optim.SGD()`

```python
optimizer = optim.SGD(model.parameters(), lr=0.01)
```

- `model.parameters()`: 학습할 weight/bias
- `lr`: learning rate, 한 번에 움직이는 크기

In [ ]:
criterion_bce = nn.BCELoss()
optimizer_bce = optim.SGD(bce_net.parameters(), lr=0.01)

print("criterion:", criterion_bce)
print("optimizer:", optimizer_bce)

BCE 방식은 모델의 출력이 이미 확률값이어야 합니다.

그래서 모델 안에 `nn.Sigmoid()`가 포함되어 있습니다.

## 10. BCELoss 방식 학습 루프

학습 루프의 기본 구조는 이전 차시와 같습니다.

```text
zero_grad → forward → loss → backward → step
```

In [ ]:
num_epochs = 2000
history_bce = np.zeros((0, 5))

for epoch in range(num_epochs):
    bce_net.train()

    optimizer_bce.zero_grad()

    outputs = bce_net(inputs)
    loss = criterion_bce(outputs, labels)

    loss.backward()
    optimizer_bce.step()

    train_loss = loss.item()
    train_pred = torch.where(outputs < 0.5, 0.0, 1.0)
    train_acc = (train_pred == labels).float().mean().item()

    bce_net.eval()
    with torch.no_grad():
        outputs_test = bce_net(inputs_test)
        loss_test = criterion_bce(outputs_test, labels_test)

        val_loss = loss_test.item()
        val_pred = torch.where(outputs_test < 0.5, 0.0, 1.0)
        val_acc = (val_pred == labels_test).float().mean().item()

    if epoch % 200 == 0:
        history_bce = np.vstack((history_bce, np.array([epoch, train_loss, train_acc, val_loss, val_acc])))

print("초기 검증 정확도:", history_bce[0, 4])
print("최종 검증 정확도:", history_bce[-1, 4])

코드 설명:

- `outputs = bce_net(inputs)`: 확률값 출력
- `criterion_bce(outputs, labels)`: BCE 손실 계산
- `torch.where(outputs < 0.5, 0, 1)`: 0.5 기준으로 class 변환
- `with torch.no_grad()`: 검증 시 gradient 계산을 끕니다.

## 11. BCELoss 학습 곡선

Loss와 Accuracy가 어떻게 변하는지 확인합니다.

In [ ]:
plt.plot(history_bce[:, 0], history_bce[:, 1], label="train loss")
plt.plot(history_bce[:, 0], history_bce[:, 3], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("BCELoss - Loss Curve")
plt.legend()
plt.show()

plt.plot(history_bce[:, 0], history_bce[:, 2], label="train acc")
plt.plot(history_bce[:, 0], history_bce[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("BCELoss - Accuracy Curve")
plt.legend()
plt.show()

그래프 해석:

- Loss가 감소하면 모델이 정답에 가까워지는 중입니다.
- Accuracy가 증가하면 맞춘 비율이 높아지는 중입니다.
- train과 validation 흐름이 비슷하면 과적합이 심하지 않다고 볼 수 있습니다.

## 12. BCEWithLogitsLoss 권장 방식

권장 방식은 모델에서 Sigmoid를 제거하고, 손실함수로 `BCEWithLogitsLoss`를 사용합니다.

구조:

```text
Linear → BCEWithLogitsLoss
```

평가할 때만 Sigmoid를 적용합니다.

### 함수 사용법: `nn.BCEWithLogitsLoss()`

```python
criterion = nn.BCEWithLogitsLoss()
loss = criterion(raw_logits, target)
```

- `raw_logits`: Sigmoid 전 점수
- `target`: 0 또는 1
- 내부적으로 Sigmoid + BCE를 안정적으로 처리

In [ ]:
class IrisLogitNet(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)

        self.l1.weight.data.fill_(1.0)
        self.l1.bias.data.fill_(1.0)

    def forward(self, x):
        x1 = self.l1(x)
        return x1

logit_net = IrisLogitNet(n_input, n_output)

criterion_logits = nn.BCEWithLogitsLoss()
optimizer_logits = optim.SGD(logit_net.parameters(), lr=0.01)

print(logit_net)
print(criterion_logits)

중요한 차이:

- `BCELoss`: 모델 출력이 확률이어야 하므로 Sigmoid 필요
- `BCEWithLogitsLoss`: 모델 출력이 raw logit이어야 하므로 Sigmoid를 모델에 넣지 않음

실무에서는 `BCEWithLogitsLoss`를 더 많이 권장합니다.

## 13. BCEWithLogitsLoss 학습 루프

BCEWithLogitsLoss에서는 예측 기준이 0.5가 아니라 logit 기준 0입니다.

```text
logit >= 0 → sigmoid(logit) >= 0.5 → class 1
logit < 0  → sigmoid(logit) < 0.5  → class 0
```

In [ ]:
num_epochs = 2000
history_logits = np.zeros((0, 5))

for epoch in range(num_epochs):
    logit_net.train()

    optimizer_logits.zero_grad()

    logits = logit_net(inputs)
    loss = criterion_logits(logits, labels)

    loss.backward()
    optimizer_logits.step()

    train_loss = loss.item()
    train_pred = torch.where(logits < 0.0, 0.0, 1.0)
    train_acc = (train_pred == labels).float().mean().item()

    logit_net.eval()
    with torch.no_grad():
        logits_test = logit_net(inputs_test)
        loss_test = criterion_logits(logits_test, labels_test)

        val_loss = loss_test.item()
        val_pred = torch.where(logits_test < 0.0, 0.0, 1.0)
        val_acc = (val_pred == labels_test).float().mean().item()

    if epoch % 200 == 0:
        history_logits = np.vstack((history_logits, np.array([epoch, train_loss, train_acc, val_loss, val_acc])))

print("초기 검증 정확도:", history_logits[0, 4])
print("최종 검증 정확도:", history_logits[-1, 4])

코드 설명:

- `logits = logit_net(inputs)`: Sigmoid 전 점수
- `criterion_logits(logits, labels)`: raw logit으로 손실 계산
- `torch.where(logits < 0.0, 0, 1)`: logit 0 기준으로 class 판단
- 확률이 필요하면 `torch.sigmoid(logits)`를 따로 사용합니다.

## 14. BCEWithLogitsLoss 학습 곡선

권장 방식의 학습 결과를 확인합니다.

In [ ]:
plt.plot(history_logits[:, 0], history_logits[:, 1], label="train loss")
plt.plot(history_logits[:, 0], history_logits[:, 3], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("BCEWithLogitsLoss - Loss Curve")
plt.legend()
plt.show()

plt.plot(history_logits[:, 0], history_logits[:, 2], label="train acc")
plt.plot(history_logits[:, 0], history_logits[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("BCEWithLogitsLoss - Accuracy Curve")
plt.legend()
plt.show()

그래프 해석:

BCEWithLogitsLoss는 Sigmoid와 BCE를 하나로 합쳐 수치적으로 안정적입니다.

이진 분류에서는 다음 패턴을 기본으로 기억하면 좋습니다.

```python
model = Linear만 출력
criterion = nn.BCEWithLogitsLoss()
pred = (logits >= 0.0).float()
prob = torch.sigmoid(logits)
```

## 15. 결정 경계 Decision Boundary

로지스틱 회귀는 feature 공간을 나누는 직선을 학습합니다.

이 직선을 **결정 경계**라고 합니다.

수식:

```text
w1*x1 + w2*x2 + b = 0
```

이를 x2에 대해 풀면:

```text
x2 = -(b + w1*x1) / w2
```

In [ ]:
bias = logit_net.l1.bias.data.numpy()
weight = logit_net.l1.weight.data.numpy()

print("BIAS:", bias)
print("WEIGHT:", weight)

def decision_boundary(x1):
    return -(bias + weight[0, 0] * x1) / weight[0, 1]

x_t0_test = x_test[y_test == 0]
x_t1_test = x_test[y_test == 1]

xl = np.array([x_test[:, 0].min(), x_test[:, 0].max()])
yl = decision_boundary(xl)

plt.scatter(x_t0_test[:, 0], x_t0_test[:, 1], marker="x", label="class 0")
plt.scatter(x_t1_test[:, 0], x_t1_test[:, 1], marker="o", label="class 1")
plt.plot(xl, yl.reshape(-1), label="decision boundary")
plt.xlabel("sepal_length")
plt.ylabel("sepal_width")
plt.title("Decision Boundary")
plt.legend()
plt.show()

그래프 해석:

- 결정 경계 위쪽/아래쪽으로 class가 나뉩니다.
- 직선이 두 그룹을 잘 나누면 분류 성능이 좋습니다.
- weight와 bias 값이 결정 경계의 위치와 기울기를 결정합니다.

## 16. 사기 거래 탐지 데이터 만들기

사기 거래 탐지는 클래스 불균형 문제가 자주 발생합니다.

예:

```text
정상 거래 95%
사기 거래 5%
```

이때 정확도만 보면 위험합니다.

모델이 전부 정상이라고 예측해도 정확도가 95%가 될 수 있기 때문입니다.

### 함수 사용법: `make_classification()`

```python
make_classification(
    n_samples=10000,
    n_features=20,
    n_classes=2,
    weights=[0.95, 0.05]
)
```

- `n_samples`: 데이터 개수
- `n_features`: feature 개수
- `n_classes`: 클래스 수
- `weights`: 클래스 비율

In [ ]:
X_fraud, y_fraud = make_classification(
    n_samples=10000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=2,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=42
)

unique, counts = np.unique(y_fraud, return_counts=True)

print("classes:", unique)
print("counts:", counts)
print("정상 거래:", counts[0])
print("사기 거래:", counts[1])
print("사기 비율:", y_fraud.mean() * 100)

함수 사용법: `np.unique()`

```python
np.unique(y, return_counts=True)
```

- 중복을 제거한 고유값과 개수를 반환합니다.
- 클래스 분포를 확인할 때 자주 사용합니다.

## 17. 사기 거래 데이터 분할과 표준화

클래스 비율을 유지하면서 train/test로 나눕니다.

표준화는 훈련 데이터로만 fit하고, 테스트 데이터에는 transform만 적용합니다.

```text
훈련 데이터: fit_transform
테스트 데이터: transform
```

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud,
    y_fraud,
    test_size=0.2,
    stratify=y_fraud,
    random_state=42
)

scaler = StandardScaler()

X_train_f_scaled = scaler.fit_transform(X_train_f)
X_test_f_scaled = scaler.transform(X_test_f)

print("훈련 데이터 사기 비율:", y_train_f.mean() * 100)
print("테스트 데이터 사기 비율:", y_test_f.mean() * 100)
print("표준화 후 평균:", X_train_f_scaled.mean())
print("표준화 후 표준편차:", X_train_f_scaled.std())

### 함수 사용법: `StandardScaler`

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

- `fit_transform`: 평균과 표준편차를 학습하고 변환합니다.
- `transform`: 이미 학습된 평균/표준편차로 변환만 합니다.
- 테스트 데이터에는 절대 `fit_transform`을 쓰지 않는 것이 좋습니다.

## 18. 사기 거래 Tensor 변환

PyTorch 학습을 위해 Tensor로 바꿉니다.

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_f_scaled)
y_train_tensor = torch.FloatTensor(y_train_f).view(-1, 1)

X_test_tensor = torch.FloatTensor(X_test_f_scaled)
y_test_tensor = torch.FloatTensor(y_test_f).view(-1, 1)

print(X_train_tensor.shape, y_train_tensor.shape)
print(X_test_tensor.shape, y_test_tensor.shape)

`torch.FloatTensor()`는 float32 Tensor를 빠르게 만드는 함수입니다.

정답은 BCEWithLogitsLoss에 맞게 `[N, 1]` 형태로 만듭니다.

## 19. 사기 거래 탐지 모델 정의

3개의 Linear Layer를 가진 간단한 MLP입니다.

구조:

```text
Linear(20 → 16) → ReLU → Linear(16 → 8) → ReLU → Linear(8 → 1)
```

마지막에는 Sigmoid를 붙이지 않습니다.

왜냐하면 `BCEWithLogitsLoss`를 사용할 것이기 때문입니다.

In [ ]:
class FraudDetectionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

fraud_model = FraudDetectionModel(input_dim=20)

print(fraud_model)

코드 설명:

- `self.fc1`, `self.fc2`, `self.fc3`: 학습되는 Linear Layer입니다.
- `self.relu`: 비선형 활성화 함수입니다.
- 마지막 출력은 raw logit입니다.

## 20. pos_weight로 클래스 불균형 보정

사기 거래가 매우 적기 때문에 class 1에 더 큰 가중치를 줍니다.

### 함수 사용법: `BCEWithLogitsLoss(pos_weight=...)`

```python
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
```

- `pos_weight`: 양성 클래스, 즉 class 1에 부여할 가중치
- 보통 `정상 개수 / 사기 개수`처럼 계산합니다.

In [ ]:
pos_weight = torch.tensor([counts[0] / counts[1]], dtype=torch.float32)

criterion_fraud = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer_fraud = optim.Adam(
    fraud_model.parameters(),
    lr=0.001,
    weight_decay=1e-3
)

print("pos_weight:", pos_weight.item())
print("criterion:", criterion_fraud)

`pos_weight`가 크면 사기 거래를 틀렸을 때 더 큰 패널티를 줍니다.

불균형 데이터에서는 정확도보다 recall, precision, F1, AUC를 함께 봐야 합니다.

## 21. 사기 거래 모델 학습

BCEWithLogitsLoss 기준 학습을 진행합니다.

In [ ]:
num_epochs = 200

history_fraud = {
    "train_loss": [],
    "test_loss": [],
    "test_acc": []
}

for epoch in range(num_epochs):
    fraud_model.train()
    optimizer_fraud.zero_grad()

    outputs = fraud_model(X_train_tensor)
    loss = criterion_fraud(outputs, y_train_tensor)

    loss.backward()
    optimizer_fraud.step()

    fraud_model.eval()
    with torch.no_grad():
        test_outputs = fraud_model(X_test_tensor)
        test_loss = criterion_fraud(test_outputs, y_test_tensor)
        test_pred = (test_outputs >= 0.0).float()
        test_acc = (test_pred == y_test_tensor).float().mean()

    history_fraud["train_loss"].append(loss.item())
    history_fraud["test_loss"].append(test_loss.item())
    history_fraud["test_acc"].append(test_acc.item())

print("최종 train loss:", history_fraud["train_loss"][-1])
print("최종 test acc:", history_fraud["test_acc"][-1])

코드 설명:

- `test_pred = (test_outputs >= 0.0).float()`: logit 기준 0 이상이면 class 1
- `test_acc`: 전체 중 맞춘 비율
- 불균형 문제에서는 accuracy만 믿으면 안 됩니다.

## 22. 사기 거래 학습 곡선

Loss와 Accuracy를 확인합니다.

In [ ]:
plt.plot(history_fraud["train_loss"], label="train loss")
plt.plot(history_fraud["test_loss"], label="test loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Fraud Detection Loss Curve")
plt.legend()
plt.show()

plt.plot(history_fraud["test_acc"], label="test accuracy")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Fraud Detection Accuracy")
plt.legend()
plt.show()

그래프 해석:

- Loss가 내려가면 학습이 진행 중입니다.
- Accuracy는 높아 보여도 사기를 잘 잡는지는 별도로 확인해야 합니다.
- 그래서 Confusion Matrix, Recall, F1, ROC-AUC를 확인합니다.

## 23. Confusion Matrix와 주요 평가 지표

### 함수 사용법

```python
confusion_matrix(y_true, y_pred)
precision_score(y_true, y_pred)
recall_score(y_true, y_pred)
f1_score(y_true, y_pred)
```

| 지표 | 의미 |
|---|---|
| Accuracy | 전체 중 맞춘 비율 |
| Precision | 사기라고 예측한 것 중 진짜 사기 비율 |
| Recall | 실제 사기 중 잡아낸 비율 |
| F1 | Precision과 Recall의 균형 |

In [ ]:
fraud_model.eval()

with torch.no_grad():
    test_logits = fraud_model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits).numpy().flatten()
    test_pred = (test_logits >= 0.0).numpy().astype(int).flatten()

y_test_np = y_test_f.astype(int)

cm = confusion_matrix(y_test_np, test_pred)

accuracy = accuracy_score(y_test_np, test_pred)
precision = precision_score(y_test_np, test_pred, zero_division=0)
recall = recall_score(y_test_np, test_pred, zero_division=0)
f1 = f1_score(y_test_np, test_pred, zero_division=0)
auc = roc_auc_score(y_test_np, test_probs)

print("Confusion Matrix:")
print(cm)

print("\nAccuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("ROC-AUC:", auc)

혼동행렬 해석:

```text
[[TN, FP],
 [FN, TP]]
```

- TN: 정상 거래를 정상으로 맞춤
- FP: 정상 거래를 사기로 잘못 예측
- FN: 사기 거래를 정상으로 놓침
- TP: 사기 거래를 사기로 맞춤

사기 탐지에서는 보통 FN, 즉 사기를 놓치는 것이 위험합니다.

## 24. ROC Curve

ROC 곡선은 임계값을 바꿨을 때 TPR과 FPR이 어떻게 변하는지 보여줍니다.

### 함수 사용법

```python
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)
```

- `y_true`: 실제 정답
- `y_prob`: class 1일 확률
- `fpr`: False Positive Rate
- `tpr`: True Positive Rate

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_np, test_probs)

plt.plot(fpr, tpr, label=f"ROC Curve (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

그래프 해석:

- 대각선은 무작위 예측입니다.
- ROC 곡선이 왼쪽 위에 가까울수록 좋습니다.
- AUC가 1에 가까울수록 좋은 분류기입니다.

## 25. Diabetes 실무형 Config 클래스

실무형 코드에서는 하이퍼파라미터를 한 곳에 모아두면 관리하기 쉽습니다.

### 함수/클래스 사용법

```python
config = Config()
config.batch_size
config.learning_rate
```

- 설정값을 객체 속성으로 저장합니다.
- 학습 코드에서 같은 값을 반복해서 쓰기 좋습니다.

In [ ]:
class Config:
    def __init__(self):
        self.test_size = 0.2
        self.val_size = 0.2
        self.random_state = 42
        self.input_dim = 10
        self.hidden_dims = [64, 32, 16]
        self.dropout_rate = 0.3
        self.batch_size = 32
        self.num_epochs = 80
        self.learning_rate = 0.001
        self.weight_decay = 0.0001
        self.patience = 15
        self.min_delta = 0.001
        self.scheduler_step_size = 20
        self.scheduler_gamma = 0.5

config = Config()

print("Batch Size:", config.batch_size)
print("Learning Rate:", config.learning_rate)

Config를 쓰면 숫자 설정을 코드 곳곳에 흩뿌리지 않아도 됩니다.

실무에서는 실험 설정을 JSON, YAML, Config 클래스로 따로 관리하는 경우가 많습니다.

## 26. Diabetes 데이터 전처리 클래스

`load_diabetes()`는 원래 회귀 데이터입니다.

여기서는 target의 중앙값보다 크면 1, 작으면 0으로 바꿔 이진 분류 문제로 만듭니다.

### 함수 사용법

```python
median = np.median(y)
y_binary = (y > median).astype(int)
```

- 중앙값보다 크면 고위험 1
- 중앙값 이하이면 저위험 0

In [ ]:
class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()

    def load_and_prepare_data(self):
        diabetes = load_diabetes()

        X = diabetes.data
        y_regression = diabetes.target

        median = np.median(y_regression)
        y = (y_regression > median).astype(int)

        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X,
            y,
            test_size=self.config.test_size,
            stratify=y,
            random_state=self.config.random_state
        )

        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val,
            y_train_val,
            test_size=self.config.val_size,
            stratify=y_train_val,
            random_state=self.config.random_state
        )

        X_train = self.scaler.fit_transform(X_train)
        X_val = self.scaler.transform(X_val)
        X_test = self.scaler.transform(X_test)

        return X_train, X_val, X_test, y_train, y_val, y_test

    def create_dataloaders(self, X_train, X_val, X_test, y_train, y_val, y_test):
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train),
            torch.FloatTensor(y_train).view(-1, 1)
        )

        val_dataset = TensorDataset(
            torch.FloatTensor(X_val),
            torch.FloatTensor(y_val).view(-1, 1)
        )

        test_dataset = TensorDataset(
            torch.FloatTensor(X_test),
            torch.FloatTensor(y_test).view(-1, 1)
        )

        train_loader = DataLoader(train_dataset, batch_size=self.config.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.config.batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=self.config.batch_size, shuffle=False)

        return train_loader, val_loader, test_loader

preprocessor = DataPreprocessor(config)

X_train_d, X_val_d, X_test_d, y_train_d, y_val_d, y_test_d = preprocessor.load_and_prepare_data()

train_loader, val_loader, test_loader = preprocessor.create_dataloaders(
    X_train_d, X_val_d, X_test_d, y_train_d, y_val_d, y_test_d
)

print("Train:", X_train_d.shape)
print("Val:", X_val_d.shape)
print("Test:", X_test_d.shape)

### 함수 사용법: `TensorDataset`과 `DataLoader`

```python
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
```

- `TensorDataset`: 입력과 정답 Tensor를 하나로 묶습니다.
- `DataLoader`: batch 단위로 데이터를 꺼냅니다.
- `shuffle=True`: 학습 데이터 순서를 섞습니다.

## 27. DiabetesClassifier 모델 정의

실무형 MLP 구조입니다.

구조:

```text
Linear → BatchNorm → ReLU → Dropout
```

을 여러 번 반복합니다.

### 함수 사용법

```python
nn.BatchNorm1d(hidden_dim)
nn.Dropout(dropout_rate)
nn.init.kaiming_normal_(weight)
```

- `BatchNorm1d`: batch 단위로 분포를 안정화
- `Dropout`: 일부 뉴런을 꺼서 과적합 완화
- `kaiming_normal_`: ReLU 계열에 적합한 weight 초기화

In [ ]:
class DiabetesClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rate=0.3):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)

        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.network(x)

diabetes_model = DiabetesClassifier(
    input_dim=config.input_dim,
    hidden_dims=config.hidden_dims,
    dropout_rate=config.dropout_rate
)

print(diabetes_model)

코드 설명:

- `layers = []`: Layer를 리스트에 차례로 담습니다.
- `nn.Sequential(*layers)`: 리스트 안 Layer를 순서대로 연결합니다.
- 마지막 출력은 logit 1개입니다.
- BCEWithLogitsLoss를 사용할 것이므로 마지막 Sigmoid는 넣지 않습니다.

## 28. EarlyStopping 클래스

EarlyStopping은 검증 손실이 더 이상 좋아지지 않으면 학습을 멈추는 기법입니다.

### 사용법

```python
early_stopping = EarlyStopping(patience=10, min_delta=0.001)
early_stopping(val_loss, model)

if early_stopping.early_stop:
    break
```

- `patience`: 몇 번까지 기다릴지
- `min_delta`: 개선으로 인정할 최소 변화량

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_loss = val_loss
            self.best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter = 0

print("EarlyStopping 준비 완료")

`__call__` 메서드가 있으면 객체를 함수처럼 호출할 수 있습니다.

```python
early_stopping(val_loss, model)
```

이렇게 쓰면 내부적으로 `early_stopping.__call__(val_loss, model)`이 실행됩니다.

## 29. Trainer 클래스

Trainer는 학습 과정을 하나의 클래스로 묶은 구조입니다.

실무에서는 학습 코드가 길어지기 때문에 다음을 클래스로 관리합니다.

- 모델
- 손실 함수
- Optimizer
- Scheduler
- EarlyStopping
- 학습 기록 history

In [ ]:
class Trainer:
    def __init__(self, model, config, device="cpu"):
        self.model = model.to(device)
        self.config = config
        self.device = device

        self.criterion = nn.BCEWithLogitsLoss()

        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )

        self.scheduler = optim.lr_scheduler.StepLR(
            self.optimizer,
            step_size=config.scheduler_step_size,
            gamma=config.scheduler_gamma
        )

        self.early_stopping = EarlyStopping(
            patience=config.patience,
            min_delta=config.min_delta
        )

        self.history = {
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
            "learning_rate": []
        }

    def train_epoch(self, train_loader):
        self.model.train()

        total_loss = 0
        correct = 0
        total = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(self.device)
            batch_y = batch_y.to(self.device)

            self.optimizer.zero_grad()

            outputs = self.model(batch_X)
            loss = self.criterion(outputs, batch_y)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item() * batch_X.size(0)

            predicted = (outputs >= 0.0).float()
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()

        return total_loss / total, correct / total

    def validate(self, val_loader):
        self.model.eval()

        total_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                outputs = self.model(batch_X)
                loss = self.criterion(outputs, batch_y)

                total_loss += loss.item() * batch_X.size(0)

                predicted = (outputs >= 0.0).float()
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()

        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader):
        for epoch in range(self.config.num_epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.validate(val_loader)

            self.scheduler.step()
            current_lr = self.optimizer.param_groups[0]["lr"]

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_acc"].append(val_acc)
            self.history["learning_rate"].append(current_lr)

            if (epoch + 1) % 20 == 0:
                print(
                    f"Epoch {epoch+1:3d} | "
                    f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                    f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
                    f"LR: {current_lr:.6f}"
                )

            self.early_stopping(val_loss, self.model)

            if self.early_stopping.early_stop:
                print("Early Stopping at Epoch", epoch + 1)
                self.model.load_state_dict(self.early_stopping.best_model_state)
                break

trainer = Trainer(diabetes_model, config, device="cpu")

print("Trainer 준비 완료")

### 함수 사용법: `StepLR`

```python
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
scheduler.step()
```

- `step_size`: 몇 epoch마다 학습률을 줄일지
- `gamma`: 학습률을 몇 배로 줄일지
- 예: 30 epoch마다 lr을 0.5배로 감소

## 30. Diabetes 모델 학습

Trainer를 사용해 학습을 실행합니다.

In [ ]:
trainer.fit(train_loader, val_loader)

출력 해석:

- `Train Loss`: 학습 데이터 손실
- `Val Loss`: 검증 데이터 손실
- `Train Acc`: 학습 데이터 정확도
- `Val Acc`: 검증 데이터 정확도
- `LR`: 현재 learning rate

검증 손실이 개선되지 않으면 EarlyStopping이 학습을 중단할 수 있습니다.

## 31. Diabetes 학습 결과 시각화

Loss, Accuracy, Learning Rate를 확인합니다.

In [ ]:
plt.plot(trainer.history["train_loss"], label="Train Loss")
plt.plot(trainer.history["val_loss"], label="Val Loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Diabetes Loss Curve")
plt.legend()
plt.show()

plt.plot(trainer.history["train_acc"], label="Train Acc")
plt.plot(trainer.history["val_acc"], label="Val Acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Diabetes Accuracy Curve")
plt.legend()
plt.show()

plt.plot(trainer.history["learning_rate"])
plt.xlabel("epoch")
plt.ylabel("learning rate")
plt.title("Learning Rate Schedule")
plt.show()

그래프 해석:

- Train Loss와 Val Loss가 함께 줄면 학습이 안정적입니다.
- Train은 좋아지는데 Val이 나빠지면 과적합을 의심합니다.
- Learning Rate 그래프는 Scheduler가 제대로 작동하는지 보여줍니다.

## 32. Diabetes 모델 평가

테스트 데이터에서 예측값, 확률값, 실제 정답을 모읍니다.

### 함수 사용법

```python
torch.sigmoid(outputs)
preds = (outputs >= 0.0).float()
```

- `torch.sigmoid(outputs)`: logit을 확률로 변환
- `(outputs >= 0.0)`: BCEWithLogitsLoss 기준 class 예측

In [ ]:
def evaluate_model(model, test_loader):
    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)

            probs = torch.sigmoid(outputs)
            preds = (outputs >= 0.0).float()

            all_preds.extend(preds.numpy())
            all_probs.extend(probs.numpy())
            all_labels.extend(batch_y.numpy())

    all_preds = np.array(all_preds).flatten()
    all_probs = np.array(all_probs).flatten()
    all_labels = np.array(all_labels).flatten()

    return all_preds, all_probs, all_labels

all_preds, all_probs, all_labels = evaluate_model(diabetes_model, test_loader)

print(classification_report(all_labels, all_preds, target_names=["Low Risk", "High Risk"]))

cm_diabetes = confusion_matrix(all_labels, all_preds)
auc_diabetes = roc_auc_score(all_labels, all_probs)

print("Confusion Matrix:")
print(cm_diabetes)
print("AUC:", auc_diabetes)

평가 지표 해석:

- Precision: 고위험이라고 예측한 것 중 실제 고위험 비율
- Recall: 실제 고위험 중 모델이 잡아낸 비율
- F1-score: Precision과 Recall의 균형
- AUC: 임계값 전체 기준 분류 성능

## 33. Precision-Recall Curve

불균형 데이터에서는 ROC보다 Precision-Recall Curve가 더 직관적일 때가 많습니다.

### 함수 사용법

```python
precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
```

- `precision`: 정밀도 배열
- `recall`: 재현율 배열
- `thresholds`: 임계값 배열

In [ ]:
precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

그래프 해석:

- Recall을 높이면 더 많은 양성을 잡지만 Precision이 떨어질 수 있습니다.
- Precision을 높이면 오탐은 줄지만 놓치는 양성이 늘어날 수 있습니다.
- 실제 서비스에서는 목적에 따라 임계값을 조절합니다.

## 34. 고객 이탈 예측용 합성 데이터 생성

고객 이탈 Churn 예측은 고객이 서비스를 떠날지 예측하는 이진 분류 문제입니다.

이번 예제는 수치형 feature와 범주형 feature가 섞인 실무형 데이터 구조를 만듭니다.

In [ ]:
N = 3000

tenure = np.random.randint(0, 72, size=N)
monthly_charges = np.round(np.random.normal(60, 15, size=N), 2)
monthly_charges = np.clip(monthly_charges, 5, 200)

total_charges = np.round(monthly_charges * (tenure + np.random.normal(0.0, 1.0, size=N)), 2)
support_calls = np.random.poisson(lam=1.8, size=N)

contract_type = np.random.choice(
    ["month-to-month", "one-year", "two-year"],
    size=N,
    p=[0.6, 0.25, 0.15]
)

has_internet = np.random.choice(["yes", "no"], size=N, p=[0.8, 0.2])
has_giga = np.random.choice(["yes", "no"], size=N, p=[0.3, 0.7])
add_on = np.random.choice(["none", "security", "backup", "both"], size=N, p=[0.5, 0.2, 0.2, 0.1])
payment_method = np.random.choice(["card", "bank", "electronic"], size=N, p=[0.35, 0.35, 0.3])

score = (
    0.04 * monthly_charges
    - 0.05 * tenure
    + 0.35 * support_calls
    + np.where(contract_type == "month-to-month", 1.2, 0.0)
    + np.where(payment_method == "electronic", 0.5, 0.0)
    + np.where(add_on == "none", 0.4, -0.2)
    + np.random.normal(0, 1, size=N)
)

prob_churn = 1 / (1 + np.exp(-(score - 3.0)))
churn = (np.random.rand(N) < prob_churn).astype(int)

df = pd.DataFrame({
    "tenure": tenure,
    "monthly_charges": monthly_charges,
    "total_charges": total_charges,
    "support_calls": support_calls,
    "contract_type": contract_type,
    "has_internet": has_internet,
    "has_giga": has_giga,
    "add_on": add_on,
    "payment_method": payment_method,
    "churn": churn
})

print(df.head())
print("\nChurn rate:", df["churn"].mean())

데이터 설명:

- `tenure`: 이용 기간
- `monthly_charges`: 월 요금
- `support_calls`: 고객센터 문의 횟수
- `contract_type`: 계약 유형
- `churn`: 이탈 여부

실무 데이터는 수치형과 범주형이 섞여 있는 경우가 많습니다.

## 35. 고객 이탈 데이터 분할

`churn` 컬럼을 정답으로 사용하고 나머지를 입력으로 사용합니다.

In [ ]:
X_churn = df.drop(columns=["churn"])
y_churn = df["churn"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn,
    y_churn,
    test_size=0.2,
    random_state=42,
    stratify=y_churn
)

print(X_train_c.shape, X_test_c.shape)
print("train churn rate:", y_train_c.mean())
print("test churn rate:", y_test_c.mean())

`stratify=y_churn`을 사용하면 train/test에서 이탈 비율이 비슷하게 유지됩니다.

분류 문제에서는 기본적으로 넣어주는 습관이 좋습니다.

## 36. ColumnTransformer와 Pipeline

수치형과 범주형 feature를 다르게 전처리합니다.

### 함수 사용법: `ColumnTransformer`

```python
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)
```

- 수치형 열에는 StandardScaler
- 범주형 열에는 OneHotEncoder

### 함수 사용법: `Pipeline`

```python
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", clf)
])
```

전처리와 모델을 하나로 묶습니다.

In [ ]:
numeric_features = ["tenure", "monthly_charges", "total_charges", "support_calls"]
categorical_features = ["contract_type", "has_internet", "has_giga", "add_on", "payment_method"]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

clf = LogisticRegression(
    class_weight="balanced",
    max_iter=200,
    solver="liblinear",
    random_state=42
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", clf)
    ]
)

print(pipe)

함수 설명:

- `handle_unknown="ignore"`: 테스트 데이터에 처음 보는 범주가 나와도 에러를 막습니다.
- `sparse_output=False`: 결과를 일반 배열 형태로 반환합니다.
- `class_weight="balanced"`: 클래스 불균형을 자동으로 보정합니다.
- `solver="liblinear"`: 작은 데이터와 이진 분류에서 안정적인 solver입니다.

## 37. 고객 이탈 모델 학습과 기본 평가

Pipeline은 `fit()` 한 번으로 전처리와 모델 학습을 함께 수행합니다.

### 함수 사용법

```python
pipe.fit(X_train, y_train)
pipe.predict_proba(X_test)[:, 1]
```

- `fit`: 전처리 학습 + 모델 학습
- `predict_proba`: class별 확률 출력
- `[:, 1]`: class 1, 즉 이탈 확률만 선택

In [ ]:
pipe.fit(X_train_c, y_train_c)

y_proba = pipe.predict_proba(X_test_c)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

acc = accuracy_score(y_test_c, y_pred)
prec = precision_score(y_test_c, y_pred, zero_division=0)
rec = recall_score(y_test_c, y_pred, zero_division=0)
f1 = f1_score(y_test_c, y_pred, zero_division=0)
auc = roc_auc_score(y_test_c, y_proba)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)
print("ROC-AUC:", auc)
print("\nClassification Report:")
print(classification_report(y_test_c, y_pred, digits=4))

결과 해석:

- 고객 이탈 문제에서는 Recall이 중요할 수 있습니다.
- 놓친 이탈 고객을 줄이고 싶다면 임계값을 낮출 수 있습니다.
- 반대로 마케팅 비용 낭비를 줄이고 싶다면 Precision을 높이는 방향으로 임계값을 조정할 수 있습니다.

## 38. 임계값 Threshold 최적화

기본 임계값은 0.5입니다.

하지만 실무에서는 목적에 따라 임계값을 조정합니다.

### 함수 사용법

```python
thresholds = np.linspace(0.1, 0.9, 81)
y_hat = (y_proba >= threshold).astype(int)
f1_score(y_test, y_hat)
```

- 여러 threshold를 시험합니다.
- F1이 가장 높은 threshold를 찾습니다.

In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)

best_thr = 0.5
best_f1 = f1

for thr in thresholds:
    y_hat = (y_proba >= thr).astype(int)
    f1_tmp = f1_score(y_test_c, y_hat, zero_division=0)

    if f1_tmp > best_f1:
        best_f1 = f1_tmp
        best_thr = thr

print("Best Threshold:", best_thr)
print("Best F1:", best_f1)

y_pred_opt = (y_proba >= best_thr).astype(int)

print("\n최적 임계값 평가")
print("Accuracy:", accuracy_score(y_test_c, y_pred_opt))
print("Precision:", precision_score(y_test_c, y_pred_opt, zero_division=0))
print("Recall:", recall_score(y_test_c, y_pred_opt, zero_division=0))
print("F1:", f1_score(y_test_c, y_pred_opt, zero_division=0))

임계값 해석:

- threshold를 낮추면 class 1 예측이 많아집니다.
- Recall은 올라가고 Precision은 떨어질 수 있습니다.
- threshold를 높이면 class 1 예측이 줄어듭니다.
- Precision은 올라가고 Recall은 떨어질 수 있습니다.

## 39. 고객 이탈 Confusion Matrix와 ROC Curve

평가 결과를 시각화합니다.

In [ ]:
cm_churn = confusion_matrix(y_test_c, y_pred_opt)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm_churn, cmap="Blues")

ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Stay", "Churn"])
ax.set_yticklabels(["Stay", "Churn"])

for (i, j), value in np.ndenumerate(cm_churn):
    ax.text(j, i, str(value), ha="center", va="center")

plt.colorbar(im)
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test_c, y_proba)
plt.title("ROC Curve - Churn")
plt.show()

그래프 해석:

- Confusion Matrix는 예측이 어디서 틀렸는지 보여줍니다.
- ROC Curve는 임계값 전체에서 모델이 얼마나 잘 구분하는지 보여줍니다.
- AUC가 높을수록 전반적인 분류 성능이 좋습니다.

## 40. Feature Importance 확인

모델이 어떤 feature를 중요하게 사용했는지 확인합니다.

### 방법 1: Logistic Regression coefficient

```python
pipe.named_steps["model"].coef_
```

- 로지스틱 회귀 계수입니다.
- 절댓값이 클수록 영향력이 큽니다.

### 방법 2: Permutation Importance

```python
permutation_importance(pipe, X_test, y_test, scoring="f1")
```

- 특정 feature를 섞었을 때 성능이 얼마나 떨어지는지 봅니다.
- 모델 전체 기준 중요도라 해석에 유용합니다.

In [ ]:
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]

num_names = numeric_features
cat_names = list(ohe.get_feature_names_out(categorical_features))

feature_names = num_names + cat_names

coef = pipe.named_steps["model"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coef
}).sort_values("coef", key=lambda s: s.abs(), ascending=False).head(10)

print("[Logistic Regression Coefficients]")
print(coef_df.to_string(index=False))

perm = permutation_importance(
    pipe,
    X_test_c,
    y_test_c,
    n_repeats=5,
    random_state=42,
    scoring="f1"
)

sorted_idx = perm.importances_mean.argsort()[::-1][:10]

plt.bar(range(len(sorted_idx)), perm.importances_mean[sorted_idx])
plt.xticks(range(len(sorted_idx)), np.array(feature_names)[sorted_idx], rotation=45, ha="right")
plt.title("Permutation Importance Top 10")
plt.tight_layout()
plt.show()

해석 주의:

- coefficient는 선형 모델 내부 가중치입니다.
- permutation importance는 실제 성능 감소 기준입니다.
- 범주형 변수는 OneHotEncoder 후 여러 개 feature로 나뉩니다.

## 41. 주요 함수 사용법 정리

| 함수 | 기본 사용법 | 인자 의미 | 언제 쓰는가 |
|---|---|---|---|
| `load_iris()` | `iris = load_iris()` | 없음 | 예제 데이터 불러오기 |
| `train_test_split()` | `train_test_split(X, y, test_size=0.2, stratify=y)` | X, y, 테스트 비율, 클래스 비율 유지 | 데이터 분리 |
| `StandardScaler()` | `scaler.fit_transform(X_train)` | 평균/표준편차 학습 후 변환 | 수치형 표준화 |
| `torch.tensor()` | `torch.tensor(x).float()` | 변환할 배열 | PyTorch 입력 준비 |
| `view(-1, 1)` | `y.view(-1, 1)` | 행 개수 자동, 열 1개 | BCE 정답 shape 맞춤 |
| `nn.Linear()` | `nn.Linear(in_features, out_features)` | 입력 차원, 출력 차원 | 선형 Layer |
| `nn.Sigmoid()` | `sigmoid(x)` | Tensor 입력 | 확률 변환 |
| `nn.BCELoss()` | `criterion(prob, y)` | 확률값, 정답 | Sigmoid 출력 모델 |
| `nn.BCEWithLogitsLoss()` | `criterion(logit, y)` | raw logit, 정답 | 권장 이진 분류 손실 |
| `optimizer.zero_grad()` | `optimizer.zero_grad()` | 없음 | gradient 초기화 |
| `loss.backward()` | `loss.backward()` | 없음 | 역전파 |
| `optimizer.step()` | `optimizer.step()` | 없음 | 파라미터 업데이트 |
| `confusion_matrix()` | `confusion_matrix(y_true, y_pred)` | 정답, 예측 class | 오답 유형 확인 |
| `precision_score()` | `precision_score(y_true, y_pred)` | 정답, 예측 class | 정밀도 |
| `recall_score()` | `recall_score(y_true, y_pred)` | 정답, 예측 class | 재현율 |
| `f1_score()` | `f1_score(y_true, y_pred)` | 정답, 예측 class | 정밀도/재현율 균형 |
| `roc_curve()` | `roc_curve(y_true, y_prob)` | 정답, 확률값 | ROC 곡선 |
| `roc_auc_score()` | `roc_auc_score(y_true, y_prob)` | 정답, 확률값 | AUC 계산 |
| `DataLoader()` | `DataLoader(dataset, batch_size=32, shuffle=True)` | Dataset, 배치 크기, 섞기 여부 | 미니배치 학습 |
| `ColumnTransformer()` | 수치/범주형 전처리 분리 | 열 이름과 변환기 | 실무형 전처리 |
| `Pipeline()` | `Pipeline([("preprocess", p), ("model", m)])` | 단계 이름과 객체 | 전처리+모델 묶기 |
| `OneHotEncoder()` | `OneHotEncoder(handle_unknown="ignore")` | 모르는 범주 처리 | 범주형 인코딩 |
| `permutation_importance()` | `permutation_importance(model, X, y)` | 모델, 데이터, 정답 | 특성 중요도 |

## 42. 시험용 요약

```text
이진 분류 = 데이터를 0 또는 1 두 그룹 중 하나로 나누는 문제
```

핵심 정리:

- 이진 분류는 두 클래스 중 하나를 예측합니다.
- Sigmoid는 선형 출력값을 0~1 사이 확률로 바꿉니다.
- Sigmoid 출력이 0.5 이상이면 class 1로 볼 수 있습니다.
- `BCELoss`는 모델 출력이 이미 확률일 때 사용합니다.
- `BCEWithLogitsLoss`는 raw logit을 입력으로 받습니다.
- `BCEWithLogitsLoss`를 쓸 때 모델 마지막에 Sigmoid를 붙이지 않습니다.
- BCEWithLogitsLoss 기준 예측은 `logit >= 0`이면 class 1입니다.
- 확률이 필요하면 평가 시점에만 `torch.sigmoid(logits)`를 적용합니다.
- Accuracy는 전체 중 맞춘 비율입니다.
- 불균형 데이터에서는 Accuracy만 보면 위험합니다.
- Precision은 양성 예측 중 실제 양성 비율입니다.
- Recall은 실제 양성 중 잡아낸 비율입니다.
- F1-score는 Precision과 Recall의 균형입니다.
- ROC-AUC는 임계값 전체에서의 분류 성능입니다.
- `pos_weight`는 소수 class에 더 큰 손실 가중치를 줍니다.
- `stratify=y`는 train/test의 class 비율을 유지합니다.
- 실무형 코드는 Config, Preprocessor, Model, Trainer, Evaluator로 나누면 관리하기 쉽습니다.
- 고객 이탈 예측처럼 수치형/범주형이 섞인 데이터는 `ColumnTransformer`와 `Pipeline`으로 처리합니다.
- 임계값 threshold는 서비스 목적에 따라 조정할 수 있습니다.